# Uzhavar AI — Agricultural RAG Knowledge Base Construction

Construction of the agricultural knowledge base for Uzhavar AI using authoritative agriculture documents, multilingual embeddings, and Qdrant Cloud.

## 1. Environment Setup

In [6]:
!pip -q install pymupdf sentence-transformers qdrant-client

In [7]:
import os
import re
import hashlib
import getpass

import numpy as np
import pymupdf

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

## 2. Configuration

In [8]:
QDRANT_URL = getpass.getpass("Qdrant Cloud URL: ").strip()
QDRANT_API_KEY = getpass.getpass("Qdrant API key: ").strip()

COLLECTION_NAME = "uzhavar_agri"
EMBEDDING_MODEL = "BAAI/bge-m3"

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
EMBED_BATCH_SIZE = 32
UPLOAD_BATCH_SIZE = 32

Qdrant Cloud URL: ··········
Qdrant API key: ··········


## 3. Knowledge Corpus

In [9]:
PDF_FILES = [
    "drip_irrigation.pdf",
    "ICAR Kharif Agro-Advisories for Farmers 2025__multi language++New.pdf",
    "Crop production.pdf",
    "PrinciplesandPracticesofWeedManagement-3rdEdition.pdf",
    "ICAR-En-Kharif-Agro-Advisories-for-Farmers-2025.pdf",
    "Farm Machinery.pdf",
]

missing = [f for f in PDF_FILES if not os.path.exists(f)]
assert not missing, f"Missing source files: {missing}"

print(f"Documents in corpus: {len(PDF_FILES)}")
for filename in PDF_FILES:
    print(f"- {filename}")

Documents in corpus: 6
- drip_irrigation.pdf
- ICAR Kharif Agro-Advisories for Farmers 2025__multi language++New.pdf
- Crop production.pdf
- PrinciplesandPracticesofWeedManagement-3rdEdition.pdf
- ICAR-En-Kharif-Agro-Advisories-for-Farmers-2025.pdf
- Farm Machinery.pdf


## 4. Document Processing

In [10]:
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_pages(pdf_path):
    document = pymupdf.open(pdf_path)
    pages = []

    try:
        for page_number, page in enumerate(document, start=1):
            text = clean_text(page.get_text("text"))
            if text:
                pages.append({
                    "page": page_number,
                    "text": text
                })
    finally:
        document.close()

    return pages

## 5. Text Chunking

In [11]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks


def build_chunks(filename):
    records = []

    for page_data in extract_pages(filename):
        page_chunks = chunk_text(page_data["text"])

        for chunk_index, text in enumerate(page_chunks):
            records.append({
                "source": filename,
                "page": page_data["page"],
                "chunk_index": chunk_index,
                "text": text,
                "region": "Tamil Nadu",
                "document_type": "agriculture"
            })

    return records

In [12]:
all_chunks = []

for filename in PDF_FILES:
    document_chunks = build_chunks(filename)
    all_chunks.extend(document_chunks)
    print(f"{filename}: {len(document_chunks)} chunks")

print(f"Total chunks: {len(all_chunks)}")
assert all_chunks, "No text chunks were generated."

drip_irrigation.pdf: 20 chunks
ICAR Kharif Agro-Advisories for Farmers 2025__multi language++New.pdf: 298 chunks
Crop production.pdf: 454 chunks
PrinciplesandPracticesofWeedManagement-3rdEdition.pdf: 192 chunks
ICAR-En-Kharif-Agro-Advisories-for-Farmers-2025.pdf: 302 chunks
Farm Machinery.pdf: 87 chunks
Total chunks: 1353


## 6. Embedding Generation

In [13]:
embedder = SentenceTransformer(EMBEDDING_MODEL)

embeddings = embedder.encode(
    [item["text"] for item in all_chunks],
    batch_size=EMBED_BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = np.asarray(embeddings, dtype=np.float32)

print(f"Embedding shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Embedding shape: (1353, 1024)


## 7. Qdrant Cloud Vector Database

In [14]:
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=300
)

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=embeddings.shape[1],
        distance=models.Distance.COSINE
    )
)

print(f"Collection ready: {COLLECTION_NAME}")

Collection ready: uzhavar_agri


## 8. Vector Ingestion

In [15]:
def point_id(source, page, chunk_index):
    value = f"{source}|{page}|{chunk_index}"
    return hashlib.sha256(value.encode("utf-8")).hexdigest()[:32]


for start in range(0, len(all_chunks), UPLOAD_BATCH_SIZE):
    end = min(start + UPLOAD_BATCH_SIZE, len(all_chunks))
    batch = []

    for index in range(start, end):
        item = all_chunks[index]

        batch.append(
            models.PointStruct(
                id=point_id(
                    item["source"],
                    item["page"],
                    item["chunk_index"]
                ),
                vector=embeddings[index].tolist(),
                payload=item
            )
        )

    for attempt in range(3):
        try:
            client.upsert(
                collection_name=COLLECTION_NAME,
                points=batch,
                wait=True,
                timeout=300
            )
            break
        except Exception:
            if attempt == 2:
                raise

    print(f"Uploaded {end}/{len(all_chunks)} vectors")

Uploaded 32/1353 vectors
Uploaded 64/1353 vectors
Uploaded 96/1353 vectors
Uploaded 128/1353 vectors
Uploaded 160/1353 vectors
Uploaded 192/1353 vectors
Uploaded 224/1353 vectors
Uploaded 256/1353 vectors
Uploaded 288/1353 vectors
Uploaded 320/1353 vectors
Uploaded 352/1353 vectors
Uploaded 384/1353 vectors
Uploaded 416/1353 vectors
Uploaded 448/1353 vectors
Uploaded 480/1353 vectors
Uploaded 512/1353 vectors
Uploaded 544/1353 vectors
Uploaded 576/1353 vectors
Uploaded 608/1353 vectors
Uploaded 640/1353 vectors
Uploaded 672/1353 vectors
Uploaded 704/1353 vectors
Uploaded 736/1353 vectors
Uploaded 768/1353 vectors
Uploaded 800/1353 vectors
Uploaded 832/1353 vectors
Uploaded 864/1353 vectors
Uploaded 896/1353 vectors
Uploaded 928/1353 vectors
Uploaded 960/1353 vectors
Uploaded 992/1353 vectors
Uploaded 1024/1353 vectors
Uploaded 1056/1353 vectors
Uploaded 1088/1353 vectors
Uploaded 1120/1353 vectors
Uploaded 1152/1353 vectors
Uploaded 1184/1353 vectors
Uploaded 1216/1353 vectors
Uploaded

## 9. Ingestion Verification

In [16]:
result = client.count(
    collection_name=COLLECTION_NAME,
    exact=True
)

stored_vectors = result.count

print(f"Collection: {COLLECTION_NAME}")
print(f"Documents processed: {len(PDF_FILES)}")
print(f"Chunks generated: {len(all_chunks)}")
print(f"Vectors stored: {stored_vectors}")

assert stored_vectors == len(all_chunks), (
    f"Vector count mismatch: expected {len(all_chunks)}, got {stored_vectors}"
)

print("Ingestion completed successfully.")

Collection: uzhavar_agri
Documents processed: 6
Chunks generated: 1353
Vectors stored: 1353
Ingestion completed successfully.


## 10. Knowledge Base Summary

The Qdrant collection contains the chunked agricultural knowledge from the six selected source documents, with source, page, chunk index, region, and document type stored as payload metadata.

The collection is ready for semantic retrieval by the Uzhavar AI RAG application.